# Maximal Marginal Relevance (MMR) Retrieval

Standard top-k similarity retrieval often returns redundant, repetitive passages from the same document section. Maximal Marginal Relevance (MMR) balances query relevance with novelty relative to already selected documents, maximizing the diversity of the context window.

## Workflow Architecture

<div align="center">
  <img src="../images/workflow_mmr.png" alt="Maximal Marginal Relevance (MMR) Retrieval Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. User Query"]):::startNode
    end
    subgraph CandidatePool["Candidate Retrieval Pool"]
        Retriever["2. Dense Vector Retrieval (FAISS)<br/>(Selects Top 20 Candidates)"]:::vsNode
    end
    subgraph MMRStage["Maximal Marginal Relevance (MMR)"]
        MMR["3. MMR Selection Loop<br/><b>Lambda * Sim(Doc, Q) - (1-Lambda) * MaxSim(Doc, Selected)</b><br/>(Balances Query Relevance vs Novelty)"]:::mmrNode
        Selected["4. Selected Diverse Context Chunks (Top 4)"]:::selNode
    end
    subgraph Out["Final Output"]
        LLM["5. Groq LLM Synthesis"]:::llmNode
        Ans(["6. Broad Non-Redundant Answer"]):::endNode
    end
    Q --> Retriever
    Retriever --> MMR
    MMR --> Selected
    Selected --> LLM
    Q --> LLM
    LLM --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef vsNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef mmrNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef selNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef llmNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Relevance vs Diversity Balance**: Lambda hyperparameter tunes between pure similarity (lambda=1) and maximal diversity (lambda=0).
- **Redundancy Suppression**: Penalizes chunks highly similar to already retrieved items.
- **Broader Coverage**: Ensures multifaceted topics get representation from distinct sections of the knowledge base.


In [1]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader(os.path.join("..", "data", "langchain_rag_dataset.txt") if os.path.exists(os.path.join("..", "data", "langchain_rag_dataset.txt")) else os.path.join("data", "langchain_rag_dataset.txt"))
raw_doc = loader.load()
raw_doc

C:\Users\itsar\AppData\Local\Temp\ipykernel_12732\336307699.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content="LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\nBM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies.\nFAISS is a high-performance library for similarity search that LangChain 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_doc)
chunks


[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4849.61it/s]


In [4]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, model)
vectorstore

In [5]:
# Use "similarity" if your documents are distinct and you just want the top matches quickly with minimal overhead.
# Use "mmr" if your documents have repeated info (e.g., versioned docs, duplicate paragraphs, repetitive logs) so the LLM gets multiple different perspectives instead of 3 nearly identical chunks.
# Use "similarity_score_threshold" if you want to prevent hallucinations when a query is out-of-domain (i.e. "say 'I don't know' if nothing matches").

retriver = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002693CA56900>, search_type='mmr', search_kwargs={'k': 3})

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.
Context: {context}
Question: {input}
""")


document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever=retriver, combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002693CA56900>, search_type='mmr', search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context provided.\nContext: {context}\nQuestion: {input}\n')
            | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_d

In [7]:
query = {"input" :"How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)
print(response['answer'])

**LangChain’s support for agents**

1. **Tool‑driven agents** – An agent in LangChain is an LLM that can decide, at run‑time, which external tool to invoke.  
   * The agent parses the user’s instruction, selects the appropriate tool (calculator, search API, custom function, database query, etc.), calls it, receives the result, and incorporates that result into its next reasoning step.  
   * Because the decision‑making logic lives inside the LLM, the agent can chain multiple tool calls in the right order to solve a complex task.

2. **External‑API and database integration** – Agents can be wired to any callable endpoint (REST APIs, GraphQL services, SQL/NoSQL databases, vector stores, etc.). The agent’s prompt template includes a description of each tool, so the LLM knows how and when to use them.

3. **Dynamic tool selection** – LangChain provides built‑in agent types (e.g., `ZeroShotAgent`, `ConversationalAgent`, `ReactAgent`) that implement popular reasoning patterns such as ReAct 